### Cleaning Project 1 data

> In this notebook we're gonna prepare the BRFSS data before applying the regression methods from `implementations.py`.
> The raw CSV files are kept unchanged and every transformation is fitted on the training data only.

Goals :
- load the train and test files with NumPy
- inspect missing values and the target balance
- replace empty values and BRFSS non-response codes
- remove the identifier column
- impute and standardize using training statistics only
- save clean arrays that can be used by the project code

> The project restricts us to NumPy and visualization libraries, so this notebook does not use Pandas or scikit-learn.
> This keeps the preprocessing consistent with the implementations we have to write ourselves.

In [2]:
import os
import numpy as np

DATA_DIR = 'dataset'
X_TRAIN_PATH = os.path.join(DATA_DIR, 'x_train.csv')
Y_TRAIN_PATH = os.path.join(DATA_DIR, 'y_train.csv')
X_TEST_PATH = os.path.join(DATA_DIR, 'x_test.csv')

np.set_printoptions(suppress=True, linewidth=120)

> We load the files directly as floats because every project feature is numeric.
> Empty cells are converted to `NaN` by NumPy, so they can be counted and imputed before training.

In [3]:
x_train_raw = np.genfromtxt(
    X_TRAIN_PATH,
    delimiter=',',
    dtype=float,
    names=True,
    missing_values='',
    filling_values=np.nan,
    autostrip=True,
)
y_train_raw = np.genfromtxt(
    Y_TRAIN_PATH,
    delimiter=',',
    dtype=float,
    names=True,
    missing_values='',
    filling_values=np.nan,
    autostrip=True,
)
x_test_raw = np.genfromtxt(
    X_TEST_PATH,
    delimiter=',',
    dtype=float,
    names=True,
    missing_values='',
    filling_values=np.nan,
    autostrip=True,
)

feature_names = np.array(x_train_raw.dtype.names)
target_name = y_train_raw.dtype.names[1]

print('x_train shape:', x_train_raw.shape)
print('y_train shape:', y_train_raw.shape)
print('x_test shape:', x_test_raw.shape)
print('number of features:', len(feature_names))
print('first feature names:', feature_names[:12])

x_train shape: (328135,)
y_train shape: (328135,)
x_test shape: (109379,)
number of features: 322
first feature names: ['Id' '_STATE' 'FMONTH' 'IDATE' 'IMONTH' 'IDAY' 'IYEAR' 'DISPCODE' 'SEQNO' '_PSU' 'CTELENUM' 'PVTRESD1']


> The target contains `-1` and `1`, which is already the convention expected by the logistic regression functions.
> We still check it explicitly so that a malformed or shifted CSV is detected before cleaning.

In [4]:
target_values = y_train_raw[target_name].astype(float)
unique_targets, target_counts = np.unique(target_values, return_counts=True)

print('target values:', unique_targets)
print('target counts:', target_counts)
print('positive class proportion:', np.mean(target_values == 1))

assert np.all(np.isin(unique_targets, [-1, 1]))
assert len(target_values) == len(x_train_raw)

target values: [-1.  1.]
target counts: [299160  28975]
positive class proportion: 0.08830207079403295


> BRFSS uses blank cells and special non-response codes such as `7`, `9`, `77`, `99`, `777`, and `999`.
> Some date and identifier columns can legitimately contain `7` or `9`, so those columns are protected from this replacement.
> Larger sentinel values are treated as missing in every feature column.

In [5]:
protected_columns = {
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR',
    'DISPCODE', 'SEQNO', '_PSU'
}
small_missing_codes = {7.0, 9.0}
large_missing_codes = {77.0, 99.0, 777.0, 999.0, 7777.0, 9999.0, 99999.0}


def convert_raw_features(raw_data, names):
    numeric_data = np.empty((len(raw_data), len(names)), dtype=float)
    numeric_data.fill(np.nan)

    for column_index, column_name in enumerate(names):
        raw_column = raw_data[column_name].astype(str)
        missing_mask = raw_column == ''
        column = np.full(len(raw_column), np.nan, dtype=float)
        valid_mask = ~missing_mask
        column[valid_mask] = raw_column[valid_mask].astype(float)

        if column_name not in protected_columns:
            missing_mask |= np.isin(column, list(small_missing_codes))
        missing_mask |= np.isin(column, list(large_missing_codes))
        numeric_data[:, column_index] = np.where(missing_mask, np.nan, column)

    return numeric_data


x_train_with_missing = convert_raw_features(x_train_raw, feature_names)
x_test_with_missing = convert_raw_features(x_test_raw, feature_names)

missing_counts = np.sum(np.isnan(x_train_with_missing), axis=0)
print('total missing training values:', int(np.sum(missing_counts)))
print('columns with the most missing values:')
for index in np.argsort(missing_counts)[-10:][::-1]:
    print(feature_names[index], int(missing_counts[index]))

total missing training values: 49213165
columns with the most missing values:
COLGHOUS 328103
LADULT 328103
PCDMDECN 327960
ASERVIST 327907
ASDRVIST 327907
ASNOSLEP 327803
ASACTLIM 327684
ASYMPTOM 327665
ASTHMED3 327663
ASINHALR 327659


> `Id` identifies a row but it is not a health feature and should not be given to the model.
> We also remove columns that contain no usable training value, because they cannot be imputed from the data.

In [7]:
keep_columns = np.array([
    column_name != 'Id' and not np.all(np.isnan(x_train_with_missing[:, index]))
    for index, column_name in enumerate(feature_names)
])

model_feature_names = feature_names[keep_columns]
x_train_model = x_train_with_missing[:, keep_columns]
x_test_model = x_test_with_missing[:, keep_columns]

print('removed columns:', int(np.sum(~keep_columns)))
print('model feature count:', x_train_model.shape[1])
print('removed feature names:', feature_names[~keep_columns])

removed columns: 1
model feature count: 321
removed feature names: ['Id']


> Missing values are replaced with the median of the corresponding training feature.
> The same training medians are applied to the test set; calculating test medians separately would leak information into the evaluation data.

In [8]:
def column_medians(data):
    medians = np.zeros(data.shape[1])
    for column_index in range(data.shape[1]):
        valid_values = data[:, column_index][~np.isnan(data[:, column_index])]
        medians[column_index] = np.median(valid_values) if len(valid_values) else 0.0
    return medians


training_medians = column_medians(x_train_model)
x_train_imputed = np.where(np.isnan(x_train_model), training_medians, x_train_model)
x_test_imputed = np.where(np.isnan(x_test_model), training_medians, x_test_model)

print('remaining missing values in train:', int(np.isnan(x_train_imputed).sum()))
print('remaining missing values in test:', int(np.isnan(x_test_imputed).sum()))

remaining missing values in train: 0
remaining missing values in test: 0


> The features have very different scales, for example encoded categories and survey measurements.
> We standardize with the training mean and standard deviation. Constant columns are kept at zero after scaling.

In [9]:
training_means = np.mean(x_train_imputed, axis=0)
training_stds = np.std(x_train_imputed, axis=0)
safe_training_stds = np.where(training_stds == 0, 1.0, training_stds)

x_train_clean = (x_train_imputed - training_means) / safe_training_stds
x_test_clean = (x_test_imputed - training_means) / safe_training_stds
y_train_clean = target_values.astype(float)

print('train mean, first five features:', np.mean(x_train_clean, axis=0)[:5])
print('train standard deviation, first five features:', np.std(x_train_clean, axis=0)[:5])
print('final train shape:', x_train_clean.shape)
print('final test shape:', x_test_clean.shape)

assert not np.isnan(x_train_clean).any()
assert not np.isnan(x_test_clean).any()

train mean, first five features: [-0.  0. -0. -0. -0.]
train standard deviation, first five features: [1. 1. 1. 1. 1.]
final train shape: (328135, 321)
final test shape: (109379, 321)


> The cleaned arrays and the statistics needed to reproduce the same transformation are saved in the `dataset` folder.
> The original CSV files are never overwritten.

In [ ]:
np.save(os.path.join(DATA_DIR, 'cleaned_x_train.npy'), x_train_clean)
np.save(os.path.join(DATA_DIR, 'cleaned_y_train.npy'), y_train_clean)
np.save(os.path.join(DATA_DIR, 'cleaned_x_test.npy'), x_test_clean)
np.savez(
    os.path.join(DATA_DIR, 'cleaning_statistics.npz'),
    feature_names=model_feature_names,
    medians=training_medians,
    means=training_means,
    standard_deviations=safe_training_stds,
)
test_ids = x_test_raw['Id'].astype(int)
np.save(os.path.join(DATA_DIR, 'test_ids.npy'), test_ids)


print('Saved cleaned arrays and training statistics.')

Saved cleaned arrays and training statistics.


### Using the cleaned data

> The arrays can now be loaded by `run.py` or by an experiment cell using the implementations from the project.

```python
x_train = np.load('dataset/cleaned_x_train.npy')
y_train = np.load('dataset/cleaned_y_train.npy')
x_test = np.load('dataset/cleaned_x_test.npy')
```

> The labels stay in `-1/1` format. For logistic regression, convert them to `0/1` only if the chosen implementation expects that convention.